# Derivation for a general integration identity involving sech

This notebook derives a general integration identity for functions involving sech and a power of a the variable of integration, i.e.

$$\int \! \xi^j \mathrm{sech(a\xi)} \, \mathrm{d\xi} .$$

The indefinite integration formula is derived through two successive applications of repeated integration by parts and the use of generalized repeated integrals and derivatives.

This notebook also serves as an example of how to easily interact with the Wolfram Language through Python.

In [1]:
# Wolfram Language Imports
from wolframclient.evaluation import WolframLanguageSession
from wolframclient.language import wl, wlexpr
ws = WolframLanguageSession()

# Display Latex
from IPython.display import display, Math


## Helper functions

Here we define some helper functions to help print pretty outputs.

In [2]:
def print_tex(expr):
    """Prints an expression in latex"""
    display(Math(expr))

def print_wlresult(expr):
    """Prints a wolfram language result in latex"""
    tex_expr = ws.evaluate(wl.ToString(wl.TeXForm(expr)))
    display(Math(tex_expr))

def print_wlexpr(expr):
    """Prints a wolfram language expression in latex"""
    tex_expr = ws.evaluate(wl.ToString(wl.TeXForm(ws.evaluate(expr))))
    display(Math(tex_expr))
    
def eval_wlprint(expr):
    """Evaluates an expression, stores, and prints the result"""
    result = ws.evaluate(expr)
    save_expr_str = 'rrr = {}'.format(expr)
    ws.evaluate(save_expr_str)
    print_wlresult(ws.evaluate('rrr'))

print('\nPrinting latex:')
print_tex('\sum_{i=1}^{10} x_i')

print('\nPrinting the result of an evaluated expression:')
print_wlresult(ws.evaluate('x+2+y+3'))

print('\nEvaluating and printing an expression directly:')
print_wlexpr('Sum[x^k, {k, 0, 5}]')

print('\nEvaluating, storing, and printing reults:')
eval_wlprint('Sum[x^k, {k, 0, 5}]')

print('\nAccessing those results in the variable "rrr":')
eval_wlprint('rrr*10 + y')


Printing latex:


<IPython.core.display.Math object>


Printing the result of an evaluated expression:


<IPython.core.display.Math object>


Evaluating and printing an expression directly:


<IPython.core.display.Math object>


Evaluating, storing, and printing reults:


<IPython.core.display.Math object>


Accessing those results in the variable "rrr":


<IPython.core.display.Math object>

## Apply repeated integration by parts

Here we apply repeated integration by parts to our integral and use Wolfram Language to check the result at each step.

The RIBP formula is given:

$$\int \! \xi^j \mathrm{sech}(a \xi) \, \mathrm{d\xi}
    = \int \! f(\xi) \frac{\partial^n g(\xi)}{\partial \xi^n} \, \mathrm{d\xi}
    = (-1)^n \int \! \frac{\partial^n f(\xi)}{\partial \xi^n} g(\xi) \, \mathrm{d\xi}
    + \sum_{k=0}^{n-1} \!
        (-1)^k \frac{\partial^k f(\xi)}{\partial \xi^k}
               \frac{\partial^{n-k-1} g(\xi)}{\partial \xi^{n-k-1}}
$$

We choose:

$$
\begin{align}
f(\xi) &= \mathrm{sech}(a\xi) \\
\frac{\partial^n g(\xi)}{\partial \xi^n} &= \xi^j \\
n &= j
\end{align}
$$

Which leads to:

$$
g(\xi) = \underbrace{\int \! \mathrm{d\xi} \, \cdots \int \! d\xi}_{j} \,
    \frac{\partial^j g(\xi)}{\partial \xi^j} =
    \underbrace{\int \! \mathrm{d\xi} \, \cdots \int \! d\xi}_{j} \, \xi^j = 
    \frac{j!}{\left(2j\right)!}\xi^{2j}
$$

In [13]:
print('Checking the definition of g(z):')

wl.Clear("Global`*")

print_wlexpr('j = 2')
print_wlexpr('f = Sech[a*x]')
print_wlexpr('g = j! / (2*j)! * x^(2*j)')

print_wlexpr('''
l = Integrate[x^j * Sech[a*x], x]
''')

print_wlexpr('''
r = (-1)^j * Integrate[ D[f, {x, j}]*g, x] + Sum[(-1)^k * D[f, {x, k}] * D[g, {x, j-k-1}], {k, 0, j-1}]
''')

# Check if the integral is equalivalent to the summation
print_wlexpr('FullSimplify[r == l]')

Checking the definition of g(z):


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

With the definition of g now verified we have:

$$\int \! \xi^j \mathrm{sech}(a \xi) \, \mathrm{d\xi}
    = (-1)^j \int \!
        \frac{\partial^j \mathrm{sech}(a\xi)}{\partial \xi^j}
        \frac{j!}{\left(2j\right)!}\xi^{2j} \,
    \mathrm{d\xi}
    + \sum_{k=0}^{j-1} \!
        (-1)^k \frac{\partial^k \mathrm{sech}(a \xi)}{\partial \xi^k}
               \frac{\partial^{j-k-1}}{\partial \xi^{j-k-1}} \left[\frac{j!}{\left(2j\right)!}\xi^{2j}\right]
$$

$$\int \! \xi^j \mathrm{sech}(a \xi) \, \mathrm{d\xi}
    = (-1)^j \frac{j!}{\left(2j\right)!} \int \!
        \frac{\partial^j \mathrm{sech}(a\xi)}{\partial \xi^j}
        \xi^{2j} \,
    \mathrm{d\xi}
    + \frac{j!}{\left(2j\right)!} \sum_{k=0}^{j-1} \!
        (-1)^k \frac{\partial^k \mathrm{sech}(a \xi)}{\partial \xi^k}
               \frac{\partial^{j-k-1}}{\partial \xi^{j-k-1}} \left[\xi^{2j}\right]
$$

The $j$th derivative of $\mathrm{sech}(a\xi)$ is:

$$
\frac{\partial^j}{\partial z^j} \mathrm{sech}(az) = 
    ia^j\left(\mathrm{Li}_{-j}(-ie^{az}) - \mathrm{Li}_{-j}(ie^{az})\right)
$$

In [14]:
print('Testing the substitutions:')

wl.Clear("Global`*")

print_wlexpr('j = 2')
print_wlexpr('f = Sech[a*x]')
print_wlexpr('g = j! / (2*j)! * x^(2*j)')

print_wlexpr('''
l = Integrate[x^j * Sech[a*x], x]
''')

print_wlexpr('''
r = (-1)^j * Integrate[ D[f, {x, j}]*g, x] + Sum[(-1)^k * D[f, {x, k}] * D[g, {x, j-k-1}], {k, 0, j-1}]
''')

# Check if the integral is equalivalent to the summation
print_wlexpr('FullSimplify[r == l]')

Testing the substitutions:


In [ ]:
ws.stop()